# PyADM1ODE-Benchmark: ein (kostenloses) LLM einbinden und bewerten

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/benchmark/llm_benchmark_de.ipynb)

Dieses Notebook zeigt den kompletten Benchmark-Durchlauf für **einen** Datenpunkt:

1. **Aufgabe laden** – eine beschriebene Biogasanlage aus dem Benchmark-Datensatz.
2. **LLM baut das Modell** – ein frei wählbares LLM erzeugt PyADM1ODE-Code.
3. **Oracle** – bei unvollständigen Aufgaben beantwortet das Oracle die Rückfragen des LLM.
4. **Bewertung** – der erzeugte Code wird ausgeführt und gegen die Referenz bewertet
   (Struktur, Maße, fehlende Werte).

Die LLM-Anbindung läuft über [**litellm**](https://github.com/BerriAI/litellm): eine
einheitliche Schnittstelle zu >100 Anbietern. Ein Anbieterwechsel ist damit ein
Einzeiler (nur die `MODEL`-Variable ändern).

Mehr Hintergrund: [Datensatz nutzen](https://dgaida.github.io/PyADM1ODE/de/benchmark/nutzung/)
und [Wie funktioniert das Oracle?](https://dgaida.github.io/PyADM1ODE/de/benchmark/oracle/)

## 1. Setup

PyADM1ODE und litellm installieren. **In Colab** diese Zelle ausführen.
**Lokal** (bereits im Repo) kann der `git clone`/`%cd`-Teil entfallen.

In [ ]:
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -q -e .
!pip install -q litellm

## 2. Anbieter & API-Key (kostenloses LLM)

Standardmäßig nutzt dieses Notebook die **kostenlose Groq-API** (großzügiges
Free-Tier). Einen Key gibt es unter <https://console.groq.com/keys>.

litellm liest den Schlüssel aus der Umgebungsvariable `GROQ_API_KEY`. Für andere
Anbieter siehe Abschnitt 7 dort steht, welche Variable jeweils gesetzt wird.

In [ ]:
import os, getpass

# Frei wählbares Modell im litellm-Format "<provider>/<model>".
MODEL = "groq/llama-3.3-70b-versatile"

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

## 3. Harness laden und einen Datenpunkt wählen

Wir verwenden die echten Benchmark-Bausteine aus `benchmark/eval/`:
`build_messages` (Prompt), `extract_code`/`extract_questions` (Antwort parsen),
`Oracle` (Rückfragen beantworten) und `evaluate_code` (ausführen + bewerten).

In [ ]:
import sys, json
from pathlib import Path

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "benchmark" / "eval"))

from prompt import SYSTEM_PROMPT, build_messages, add_oracle_answers, append_assistant
from solve import extract_code, extract_questions
from oracle import Oracle
from runner import evaluate_code

DATASET = REPO / "benchmark" / "dataset"
index = json.loads((DATASET / "index.json").read_text(encoding="utf-8"))

# Tipp: zur Übersicht alle verfügbaren Datenpunkte auflisten
for e in index["datapoints"]:
    print(f"{e['id']:<22} {e['regime']:<16} {e['modality']:<7} {e['language']}")

In [ ]:
# Einen Datenpunkt auswählen.
# 'BGA1_terse_de' ist unvollständig (underspecified) -> aktiviert das Oracle.
DP_ID = "BGA1_terse_de"

entry = next(e for e in index["datapoints"] if e["id"] == DP_ID)
dp_path = DATASET / entry["path"]
dp = json.loads(dp_path.read_text(encoding="utf-8"))
dp_dir = str(dp_path.parent)

print("ID:      ", DP_ID)
print("Regime:  ", dp["regime"])
print("Modalität:", dp["input"]["modality"], "| Sprache:", dp["input"]["language"])
print("\nBeschreibung:\n", dp["input"].get("content", "(nur Bild)"))

## 4. LLM-Anbindung über litellm

Eine einzige Funktion `call_llm` kapselt den Anbieter. Der System-Prompt (komplette
PyADM1ODE-API-Kurzreferenz) wird vorangestellt. Text-Inhalte werden zu einem String
zusammengefasst (maximale Anbieter-Kompatibilität). Bildinhalte bleiben als
OpenAI-Vision-Liste erhalten, sodass Vision-Modelle Skizzen-Datenpunkte lesen können.

In [ ]:
import litellm

def _to_litellm(messages):
    """prompt.py liefert Inhalte als Liste von Parts. Reine Text-Inhalte zu einem
    String zusammenfassen. image_url-Parts unverändert lassen (Vision)."""
    norm = []
    for m in messages:
        c = m["content"]
        if isinstance(c, list) and all(p.get("type") == "text" for p in c):
            c = "\n".join(p["text"] for p in c)
        norm.append({"role": m["role"], "content": c})
    return norm

def call_llm(messages, model=MODEL, max_tokens=4096):
    full = [{"role": "system", "content": SYSTEM_PROMPT}] + _to_litellm(messages)
    resp = litellm.completion(model=model, messages=full, max_tokens=max_tokens, temperature=0.0)
    return resp.choices[0].message.content

## 5. Ablauf: Runde 1 → Oracle → Runde 2

Dieselbe Logik wie in `benchmark/eval/solve.py`: Das LLM darf bei unvollständigen
Aufgaben zuerst Rückfragen stellen. Stellt es welche (statt sofort Code zu liefern),
beantwortet das Oracle sie, und das LLM schreibt in Runde 2 den fertigen Code.

In [ ]:
regime = dp.get("regime", "underspecified")
allow_questions = regime == "underspecified"

messages = build_messages(dp, dp_dir, allow_questions=allow_questions)
resp1 = call_llm(messages)
print("=== Runde 1 (LLM) ===\n", resp1[:1000])

code_str = extract_code(resp1)
questions = extract_questions(resp1)
oracle_turns = 0

if code_str is None and questions and allow_questions:
    oracle_turns = 1
    answer_text = Oracle(dp).answer(questions)
    print("\n=== Oracle antwortet ===\n", answer_text)
    append_assistant(messages, resp1)
    add_oracle_answers(messages, answer_text)
    resp2 = call_llm(messages)
    print("\n=== Runde 2 (LLM) ===\n", resp2[:1000])
    code_str = extract_code(resp2)

assert code_str, "Kein Python-Code in der LLM-Antwort gefunden."
print("\nOracle-Runden:", oracle_turns)
print("\n--- Generierter Code ---\n", code_str)

## 6. Modell ausführen und bewerten

`evaluate_code` führt den Code **isoliert** (Subprozess + Timeout) aus, serialisiert
die gebaute Anlage und vergleicht sie mit der Referenz. Bewertet werden drei Scores:

- **Struktur** – Bauteile (nach Typ) und Verbindungen.
- **Maße** – simulierte Parameter (`V_liq`, `V_gas`, `T_ad`, …) im Akzeptanzband.
- **Fehlende Werte** – erfragt oder plausibel gefüllt statt still erfunden.

Die vom LLM gestellten Fragen geben wir als `response` weiter – das fließt in den
Lücken-Score ein.

In [ ]:
response = {"open_questions": questions or []}
report = evaluate_code(dp, code_str, response)

print(report.pretty())
print("\nScores:", {
    "build_success": report.build_success,
    "structure": round(report.structure, 3),
    "measures": round(report.measures, 3),
    "gaps": round(report.gaps, 3),
    "overall": report.overall(),
})

## 7. Ein anderes (freies) LLM einbinden

Dank litellm genügt es, oben `MODEL` (und ggf. den passenden API-Key) zu ändern –
`call_llm` bleibt gleich. Einige Optionen mit kostenlosem Zugang:

| Anbieter | `MODEL` (Beispiel) | API-Key (Umgebungsvariable) |
| -------- | ------------------ | --------------------------- |
| Groq (Free-Tier) | `groq/llama-3.3-70b-versatile` | `GROQ_API_KEY` |
| Groq Vision (Skizzen) | `groq/meta-llama/llama-4-scout-17b-16e-instruct` | `GROQ_API_KEY` |
| Google Gemini (Free-Tier) | `gemini/gemini-2.0-flash` | `GEMINI_API_KEY` |
| OpenRouter (Free-Modelle) | `openrouter/<modell>:free` | `OPENROUTER_API_KEY` |
| Ollama (lokal, ohne Key) | `ollama/llama3` | – |

> Modellnamen ändern sich; die aktuelle Liste steht beim jeweiligen Anbieter bzw. in
> der [litellm-Doku](https://docs.litellm.ai/docs/providers).

**Für Bild-/Hybrid-Datenpunkte** (z. B. `BGA1_sketch`) ein **Vision-Modell** wählen –
die Bildanbindung übernimmt `_to_litellm` bereits.

**Ganzen Datensatz auswerten:** Für einen Lauf über alle 24 Datenpunkte gibt es das
CLI-Skript `python benchmark/eval/solve.py` (siehe *Datensatz nutzen*).